# Generador de PVC y recursos

Este notebook ayuda a construir YAML inicial para `PersistentVolumeClaim` y bloques de `resources`. Sirve para prototipos y para practicar dimensionamiento básico.


In [ ]:
from textwrap import dedent


def render_pvc(name, storage='1Gi', access_mode='ReadWriteOnce', storage_class='demo-manual'):
    return dedent(
        f"""
        apiVersion: v1
        kind: PersistentVolumeClaim
        metadata:
          name: {name}
        spec:
          accessModes:
            - {access_mode}
          storageClassName: {storage_class}
          resources:
            requests:
              storage: {storage}
        """
    ).strip()


def render_resources(cpu_request='100m', memory_request='128Mi', cpu_limit='300m', memory_limit='256Mi'):
    return dedent(
        f"""
        resources:
          requests:
            cpu: {cpu_request}
            memory: {memory_request}
          limits:
            cpu: {cpu_limit}
            memory: {memory_limit}
        """
    ).strip()


def estimate_total_requests(replicas, cpu_request_m='100m', memory_request='128Mi'):
    cpu = int(cpu_request_m[:-1]) * replicas if cpu_request_m.endswith('m') else float(cpu_request_m) * 1000 * replicas
    memory = int(memory_request[:-2]) * replicas if memory_request.endswith('Mi') else int(float(memory_request[:-2]) * 1024 * replicas)
    return {'total_cpu_m': cpu, 'total_memory_mib': memory}


In [ ]:
pvc_yaml = render_pvc('app-pvc', storage='2Gi')
resources_yaml = render_resources(cpu_request='150m', memory_request='192Mi', cpu_limit='500m', memory_limit='384Mi')
totals = estimate_total_requests(replicas=3, cpu_request_m='150m', memory_request='192Mi')

print(pvc_yaml)
print()
print(resources_yaml)
print()
print('Totales estimados:', totals)


## Ideas para practicar

- Cambia el tamaño del claim y el modo de acceso.
- Estima requests para 5 o 10 réplicas.
- Usa el resultado para crear un nuevo ejemplo en `examples/k8s/`.
